# Mini-Tritty: test and export a trained ternary model

This notebook **does not train anything**. It takes your trained `ternary_full_best.pt` and:

1. **Tests** it (validation loss, and how much an int8 embedding table costs)
2. **Exports** it to one compact binary file (`bitlm_ternary.bin`, ternary weights packed 4 per byte)
3. **Checks the file** with a pure-NumPy engine that uses only the `.bin` (integer math + KV cache, exactly what the Rust port will do) and compares its logits with PyTorch
4. Saves **test vectors** (inputs plus PyTorch's answers) for testing the Rust code on your PC

**Setup**
- Sidebar -> **Add Input** -> attach your Dataset containing `ternary_full_best.pt` and `tokenizer.json`.
- **Accelerator: None** is fine (a GPU only makes the loss test faster). **Internet: On** is needed only if `val.bin` is not in your Dataset (it is then rebuilt from TinyStories).
- Run the cells top to bottom. Nothing here takes more than a few minutes.

## 1. Setup and locate your files

In [ ]:
!pip install -q tokenizers datasets

import os, math, time, json, struct, copy, glob, contextlib, zipfile, shutil
from dataclasses import dataclass
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

WORK_DIR  = os.environ.get("WORK_DIR", "/kaggle/working")
INPUT_DIR = os.environ.get("INPUT_DIR", "/kaggle/input")
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)

def find_file(name):
    """Look in the working folder first, then in every attached Dataset."""
    p = f"{WORK_DIR}/{name}"
    if os.path.exists(p):
        return p
    hits = glob.glob(f"{INPUT_DIR}/**/{name}", recursive=True)
    return hits[0] if hits else None

CKPT = find_file("ternary_full_best.pt")
TOK  = find_file("tokenizer.json")
print("device:     ", device)
print("checkpoint: ", CKPT)
print("tokenizer:  ", TOK)
assert CKPT and TOK, "Files not found: use Add Input to attach the Dataset that holds ternary_full_best.pt and tokenizer.json"

## 2. Tokenizer and validation data

In [ ]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(TOK)
EOS = "<|endoftext|>"
EOS_ID = tok.token_to_id(EOS)
VOCAB_SIZE = tok.get_vocab_size()
print("vocab size:", VOCAB_SIZE, "| EOS id:", EOS_ID)

# Same sentence as in the training notebook. For your tokenizer this should print:
# [432, 447, 259, 397, 12, 403, 282, 259, 400, 450, 498, 359, 14]
print(tok.encode("Once upon a time, there was a little girl named Lily.").ids)

val_path = find_file("val.bin")
if val_path is None:
    from datasets import load_dataset
    print("val.bin not found, rebuilding it from the TinyStories validation split...")
    vds = load_dataset("roneneldan/TinyStories", split="validation[:5000]")
    ids = []
    for i in range(0, len(vds), 1000):
        for e in tok.encode_batch(vds[i:i + 1000]["text"]):
            ids.extend(e.ids)
            ids.append(EOS_ID)
    val_path = f"{WORK_DIR}/val.bin"
    np.array(ids, dtype=np.uint16).tofile(val_path)

val_data = np.memmap(val_path, dtype=np.uint16, mode="r")
print("validation tokens:", f"{len(val_data):,}", "(expect 1,122,050 for the standard setup)")

## 3. The model definition

This must match the training notebook exactly, otherwise the checkpoint will not load. It is the same ternary model (BitLinear everywhere, extra norms in front of `wo` and `w_down`).

In [ ]:
# MODEL DEFINITION
@dataclass
class Config:
    vocab_size: int = 2048
    block_size: int = 256          # context length
    n_layer: int = 6
    n_head: int = 8
    dim: int = 256
    hidden_dim: int = 704
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5

cfg = Config(vocab_size=VOCAB_SIZE)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x32 = x.float()
        out = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (out * self.weight.float()).to(x.dtype)

def precompute_rope(head_dim, max_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_len).float()
    angles = torch.outer(t, freqs)
    return angles.cos(), angles.sin()

def apply_rope(x, cos, sin):
    T = x.shape[2]
    cos = cos[:T][None, None].to(x.dtype)
    sin = sin[:T][None, None].to(x.dtype)
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

class BitLinear(nn.Linear):
    """Ternary weights (absmean) + int8 per-token activations (absmax), BitNet b1.58 style."""
    def __init__(self, in_features, out_features, norm=False):
        super().__init__(in_features, out_features, bias=False)
        self.norm = RMSNorm(in_features) if norm else None

    @staticmethod
    def weight_quant(w):
        gamma = w.abs().mean().clamp(min=1e-5)
        w_q = (w / gamma).round().clamp(-1, 1)
        return w_q, gamma

    @staticmethod
    def act_quant(x):
        s = 127.0 / x.abs().amax(dim=-1, keepdim=True).clamp(min=1e-5)
        x_q = (x * s).round().clamp(-128, 127)
        return x_q, s

    def forward(self, x):
        if self.norm is not None:
            x = self.norm(x)
        w = self.weight
        w_q, gamma = self.weight_quant(w)
        x32 = x.float()
        x_q, s = self.act_quant(x32)
        w_ste = w   + (w_q * gamma - w).detach()
        x_ste = x32 + (x_q / s     - x32).detach()
        return F.linear(x_ste, w_ste)

class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.dim // cfg.n_head
        self.wq = BitLinear(cfg.dim, cfg.dim)
        self.wk = BitLinear(cfg.dim, cfg.dim)
        self.wv = BitLinear(cfg.dim, cfg.dim)
        self.wo = BitLinear(cfg.dim, cfg.dim, norm=True)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        def split_heads(t):
            return t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k, v = split_heads(self.wq(x)), split_heads(self.wk(x)), split_heads(self.wv(x))
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.wo(y.transpose(1, 2).contiguous().view(B, T, C))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = BitLinear(cfg.dim, cfg.hidden_dim)
        self.w_up   = BitLinear(cfg.dim, cfg.hidden_dim)
        self.w_down = BitLinear(cfg.hidden_dim, cfg.dim, norm=True)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.attn = Attention(cfg)
        self.mlp_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.mlp = MLP(cfg)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.attn_norm(x), cos, sin)
        x = x + self.mlp(self.mlp_norm(x))
        return x

class Transformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.dim)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.final_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.dim, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight          # weight tying
        cos, sin = precompute_rope(cfg.dim // cfg.n_head, cfg.block_size, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x = blk(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.final_norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

m = Transformer(cfg).to(device)
raw = torch.load(CKPT, map_location=device)
raw = raw["model"] if "model" in raw else raw          # accepts a best-weights file or a resume checkpoint
m.load_state_dict(raw)
m.eval()
n_params = sum(p.numel() for p in m.parameters())
print(f"loaded OK: {n_params:,} parameters (expect 5,350,272)")

## 4. Tests before exporting

- **Test 1:** the model's validation loss (about 1.70 for your 30k-step ternary model).
- **Test 2:** replace the embedding table with an **int8** version (one scale per token row) and measure how much the loss changes. If it costs more than about 0.02, tell me before exporting.
- **Test 3:** how many ternary weights are zero (zeros can be skipped in the ESP32 kernel).

In [ ]:
BATCH_SIZE, N_EVAL = 64, (40 if device == "cuda" else 8)

def get_val_batch():
    ix = torch.randint(len(val_data) - cfg.block_size - 1, (BATCH_SIZE,))
    x = torch.stack([torch.from_numpy(val_data[i : i + cfg.block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(val_data[i + 1 : i + 1 + cfg.block_size].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def eval_loss(model, batches, fp16=False):
    model.eval()
    tot = 0.0
    for x, y in batches:
        ctx = torch.autocast(device_type="cuda", dtype=torch.float16) if fp16 else contextlib.nullcontext()
        with ctx:
            _, loss = model(x, y)
        tot += loss.item()
    return tot / len(batches)

torch.manual_seed(0)
batches = [get_val_batch() for _ in range(N_EVAL)]      # the same batches for every test below
print(f"(using {N_EVAL} batches of {BATCH_SIZE} x {cfg.block_size} tokens)\n")

# Test 1
loss_fp32 = eval_loss(m, batches)
loss_fp16 = eval_loss(m, batches, True) if device == "cuda" else None
print(f"[1] val loss, fp32 math:      {loss_fp32:.4f}")
if loss_fp16 is not None:
    print(f"    val loss, fp16 autocast:  {loss_fp16:.4f}   (how it was measured during training)")

# Test 2
def quant_embedding(E):
    scale = E.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / 127.0
    q = torch.round(E / scale).clamp(-127, 127).to(torch.int8)
    return q, scale.squeeze(1)

E = m.tok_emb.weight.data.float().cpu()
emb_q, emb_scale = quant_embedding(E)
deq = emb_q.float() * emb_scale[:, None]

mq = copy.deepcopy(m)                                   # copy of the model that uses the int8 embedding
mq.tok_emb.weight.data.copy_(deq.to(device))
assert mq.lm_head.weight.data_ptr() == mq.tok_emb.weight.data_ptr(), "weight tying broke"
loss_int8 = eval_loss(mq, batches)
print(f"[2] val loss, int8 embedding: {loss_int8:.4f}   (change: {loss_int8 - loss_fp32:+.4f})")
print(f"    max embedding error: {(E - deq).abs().max().item():.5f}")

# Test 3
zeros = total = 0
for mod in m.modules():
    if isinstance(mod, BitLinear):
        wq, _ = BitLinear.weight_quant(mod.weight.data.float())
        zeros += (wq == 0).sum().item()
        total += wq.numel()
print(f"[3] ternary weights: {total:,} | zeros: {zeros / total * 100:.1f}%")

## 5. Export to a binary file

**File layout** (all little-endian, every section 4-byte aligned):

| Section | Contents |
|---|---|
| Header, 64 bytes | magic `BTLM`, version, vocab, context, layers, heads, dim, hidden_dim (u32), rope_theta, norm_eps (f32), zero padding |
| Embedding | f32 scale per token row, then int8 values (vocab x dim) |
| Final norm | f32 x dim |
| Per layer | f32 norms: attn_norm, wo_norm, mlp_norm (dim each), down_norm (hidden_dim); then 7 ternary matrices in the order wq, wk, wv, wo, w_gate, w_up, w_down. Each = f32 `gamma` + weights packed 4 per byte, row by row (out x in) |

**2-bit packing:** each weight is stored as `w + 1` (-1 -> 0, 0 -> 1, +1 -> 2), and the first of four weights in a byte sits in the lowest two bits.

In [ ]:
def pack_ternary(wq):                       # wq: int8 array (out, in) with values -1/0/+1
    out_f, in_f = wq.shape
    assert in_f % 4 == 0
    c = (wq + 1).astype(np.uint8).reshape(out_f, in_f // 4, 4)
    return (c[..., 0] | (c[..., 1] << 2) | (c[..., 2] << 4) | (c[..., 3] << 6)).astype(np.uint8)

sd = {k: v.detach().float().cpu() for k, v in m.state_dict().items()}
out = bytearray()

def put_f32(t):
    out.extend(np.ascontiguousarray(t.numpy().astype("<f4")).tobytes())

header = b"BTLM" + struct.pack("<7I", 1, cfg.vocab_size, cfg.block_size, cfg.n_layer,
                                cfg.n_head, cfg.dim, cfg.hidden_dim)
header += struct.pack("<2f", cfg.rope_theta, cfg.norm_eps)
out.extend(header.ljust(64, b"\0"))

put_f32(emb_scale)                                         # f32[vocab]
out.extend(emb_q.numpy().astype(np.int8).tobytes())        # int8[vocab * dim]
put_f32(sd["final_norm.weight"])                           # f32[dim]

for i in range(cfg.n_layer):
    p = f"blocks.{i}."
    put_f32(sd[p + "attn_norm.weight"])
    put_f32(sd[p + "attn.wo.norm.weight"])
    put_f32(sd[p + "mlp_norm.weight"])
    put_f32(sd[p + "mlp.w_down.norm.weight"])
    for name in ["attn.wq", "attn.wk", "attn.wv", "attn.wo", "mlp.w_gate", "mlp.w_up", "mlp.w_down"]:
        wq, gamma = BitLinear.weight_quant(sd[p + name + ".weight"])
        out.extend(struct.pack("<f", gamma.item()))
        out.extend(pack_ternary(wq.to(torch.int8).numpy()).tobytes())

bin_path = f"{WORK_DIR}/bitlm_ternary.bin"
with open(bin_path, "wb") as f:
    f.write(out)
print(f"wrote {bin_path}")
print(f"size: {len(out):,} bytes = {len(out) / 1e6:.2f} MB   (expect about 1.75 MB)")

## 6. Reference engine: reads only the `.bin`, integer math, KV cache

No PyTorch and no checkpoint are used here. It unpacks the 2-bit weights from the file and processes **one token at a time**, which is exactly what the Rust code on the ESP32 will do. Then we compare its logits with PyTorch's.

In [ ]:
class RefModel:
    def __init__(self, path):
        buf = open(path, "rb").read()
        assert buf[:4] == b"BTLM", "bad magic"
        ver, V, T, nl, nh, D, H = struct.unpack_from("<7I", buf, 4)
        theta, eps = struct.unpack_from("<2f", buf, 32)
        assert ver == 1
        self.V, self.T, self.nl, self.nh, self.D, self.H = V, T, nl, nh, D, H
        self.hd = D // nh
        self.eps = np.float32(eps)
        pos = [64]

        def f32(n):
            a = np.frombuffer(buf, dtype="<f4", count=n, offset=pos[0]).astype(np.float32)
            pos[0] += 4 * n
            return a
        def i8(n):
            a = np.frombuffer(buf, dtype=np.int8, count=n, offset=pos[0])
            pos[0] += n
            return a
        def tern(out_f, in_f):
            gamma = np.float32(f32(1)[0])
            nb = out_f * in_f // 4
            packed = np.frombuffer(buf, dtype=np.uint8, count=nb, offset=pos[0]).reshape(out_f, in_f // 4)
            pos[0] += nb
            codes = (packed[:, :, None] >> np.array([0, 2, 4, 6], dtype=np.uint8)) & 3
            w = codes.reshape(out_f, in_f).astype(np.int32) - 1      # back to -1 / 0 / +1
            return w, gamma

        self.emb_scale = f32(V)
        self.emb_f = i8(V * D).reshape(V, D).astype(np.float32)
        self.final_norm = f32(D)
        self.layers = []
        for _ in range(nl):
            L = {"attn_norm": f32(D), "wo_norm": f32(D), "mlp_norm": f32(D), "down_norm": f32(H)}
            for name, (o, i) in [("wq", (D, D)), ("wk", (D, D)), ("wv", (D, D)), ("wo", (D, D)),
                                 ("w_gate", (H, D)), ("w_up", (H, D)), ("w_down", (D, H))]:
                L[name] = tern(o, i)
            self.layers.append(L)
        assert pos[0] == len(buf), f"{len(buf) - pos[0]} unread bytes: reader and writer disagree"

        freqs = (1.0 / (theta ** (np.arange(0, self.hd, 2, dtype=np.float32) / self.hd))).astype(np.float32)
        ang = np.outer(np.arange(T, dtype=np.float32), freqs).astype(np.float32)
        self.cos, self.sin = np.cos(ang).astype(np.float32), np.sin(ang).astype(np.float32)
        self.reset()

    def reset(self):
        self.kc = np.zeros((self.nl, self.T, self.nh, self.hd), np.float32)    # KV cache
        self.vc = np.zeros_like(self.kc)

    def rmsnorm(self, x, w):
        r = np.float32(1.0) / np.sqrt(np.mean(x * x, dtype=np.float32) + self.eps)
        return (x * r) * w

    @staticmethod
    def bitlinear(x, wg):
        w, gamma = wg
        amax = max(np.abs(x).max(), np.float32(1e-5))
        s = np.float32(127.0) / amax                                  # per-token scale
        xq = np.clip(np.round(x * s), -128, 127).astype(np.int32)     # int8 activations
        acc = w @ xq                                                  # integer matmul
        return acc.astype(np.float32) * (gamma / s)                   # one rescale at the end

    def step(self, tok, pos):
        nh, hd = self.nh, self.hd
        half = hd // 2
        c, s_ = self.cos[pos], self.sin[pos]
        def rope(t):                                                  # t: (nh, hd), split-half pairing
            t1, t2 = t[:, :half], t[:, half:]
            return np.concatenate([t1 * c - t2 * s_, t1 * s_ + t2 * c], axis=1)

        x = self.emb_f[tok] * self.emb_scale[tok]
        for li, L in enumerate(self.layers):
            h = self.rmsnorm(x, L["attn_norm"])
            q = rope(self.bitlinear(h, L["wq"]).reshape(nh, hd))
            k = rope(self.bitlinear(h, L["wk"]).reshape(nh, hd))
            v = self.bitlinear(h, L["wv"]).reshape(nh, hd)
            self.kc[li, pos], self.vc[li, pos] = k, v
            K, V = self.kc[li, :pos + 1], self.vc[li, :pos + 1]       # all tokens so far
            sc = np.einsum("hd,thd->ht", q, K) / np.float32(np.sqrt(hd))
            sc -= sc.max(axis=1, keepdims=True)
            p = np.exp(sc)
            p /= p.sum(axis=1, keepdims=True)
            y = np.einsum("ht,thd->hd", p, V).reshape(-1)
            x = x + self.bitlinear(self.rmsnorm(y, L["wo_norm"]), L["wo"])
            h = self.rmsnorm(x, L["mlp_norm"])
            g, u = self.bitlinear(h, L["w_gate"]), self.bitlinear(h, L["w_up"])
            a = (g / (np.float32(1.0) + np.exp(-g))) * u              # SiLU(g) * u
            x = x + self.bitlinear(self.rmsnorm(a, L["down_norm"]), L["w_down"])
        x = self.rmsnorm(x, self.final_norm)
        return (self.emb_f @ x) * self.emb_scale                      # tied output layer

ref = RefModel(bin_path)
print(f"file parsed: {ref.nl} layers, dim {ref.D}, vocab {ref.V}, context {ref.T}")

# Compare with PyTorch (fp32, int8-embedding model) on 48 real tokens
ids48 = val_data[:48].astype(np.int64).tolist()
mq_cpu = copy.deepcopy(mq).cpu().eval()
with torch.no_grad():
    t_logits = mq_cpu(torch.tensor([ids48]))[0][0].numpy()            # (48, vocab)

ref.reset()
t0 = time.time()
r_logits = np.stack([ref.step(t, i) for i, t in enumerate(ids48)])
d = np.abs(t_logits - r_logits)
agree = float((t_logits.argmax(1) == r_logits.argmax(1)).mean())
print(f"({time.time() - t0:.1f}s for 48 tokens)")
print(f"max  |diff|: {d.max():.5f}")
print(f"mean |diff|: {d.mean():.6f}   (typical logit size: {np.abs(t_logits).mean():.2f})")
print(f"argmax agreement: {agree * 100:.1f}%")

## 7. Generate text with the reference engine, and save test vectors for Rust

The sampler (temperature, top-k, repetition penalty) is written in plain NumPy on purpose: it is what you will port to Rust.

**Test vector file:** `u32 n`, `u32 vocab`, `i32 ids[n]`, `f32 logits[n][vocab]`. Feed the ids to your Rust model one at a time and compare the logits after each token.

In [ ]:
def sample_next(logits, history, rng, temperature=0.7, top_k=40, rep_penalty=1.15, window=64):
    l = logits.astype(np.float32).copy()
    recent = np.unique(history[-window:])
    v = l[recent]
    l[recent] = np.where(v > 0, v / rep_penalty, v * rep_penalty)
    l /= temperature
    k = min(top_k, len(l))
    kth = np.partition(l, -k)[-k]
    l[l < kth] = -np.inf
    p = np.exp(l - l.max()).astype(np.float64)
    p /= p.sum()
    return int(rng.choice(len(p), p=p))

def ref_generate(prompt, max_new=150, seed=1):
    rng = np.random.default_rng(seed)
    ids = tok.encode(prompt).ids
    ref.reset()
    for i, t in enumerate(ids):
        logits = ref.step(t, i)
    n = 0
    while len(ids) < ref.T and n < max_new:
        nxt = sample_next(logits, ids, rng)
        if nxt == EOS_ID:
            break
        logits = ref.step(nxt, len(ids))
        ids.append(nxt)
        n += 1
    return tok.decode(ids)

for p in ["Once upon a time, there was a little girl named", "One day, a boy named Tom found a"]:
    print(ref_generate(p), "\n" + "-" * 60)

n_tv = 16
tv_path = f"{WORK_DIR}/test_vectors.bin"
with open(tv_path, "wb") as f:
    f.write(struct.pack("<2I", n_tv, ref.V))
    f.write(np.array(ids48[:n_tv], dtype="<i4").tobytes())
    f.write(t_logits[:n_tv].astype("<f4").tobytes())
print(f"\nsaved {tv_path} ({os.path.getsize(tv_path):,} bytes)")

## 8. Bundle everything for download

Creates `export_bundle.zip` (the model file, the test vectors, the tokenizer and a small results file for your write-up). Download it from the Output panel, or use **Save Version** and then **New Dataset from Output**.

In [ ]:
results = {
    "params": n_params,
    "val_loss_fp32": loss_fp32,
    "val_loss_fp16_autocast": loss_fp16,
    "val_loss_int8_embedding": loss_int8,
    "ternary_zero_fraction": zeros / total,
    "bin_bytes": len(out),
    "ref_vs_pytorch_max_abs_diff": float(d.max()),
    "ref_vs_pytorch_mean_abs_diff": float(d.mean()),
    "ref_vs_pytorch_argmax_agreement": agree,
}
res_path = f"{WORK_DIR}/export_results.json"
with open(res_path, "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

zip_path = f"{WORK_DIR}/export_bundle.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(bin_path, "bitlm_ternary.bin")
    z.write(tv_path, "test_vectors.bin")
    z.write(res_path, "export_results.json")
    z.write(TOK, "tokenizer.json")
print(f"\n{zip_path}: {os.path.getsize(zip_path) / 1e6:.2f} MB")